In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_classic.chains import RetrievalQA


C:\Users\Nihala\AppData\Local\Temp\ipykernel_14100\2898870174.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
# 1. LOADING TEXT FILE
loader = TextLoader("knowledge_base.txt", encoding="utf-8")
documents = loader.load()

documents

[Document(metadata={'source': 'knowledge_base.txt'}, page_content="================================================================================\nCOURSE NAME: KTET (Kerala Teacher Eligibility Test)\n================================================================================\n\nCategory:\nTeaching Eligibility\n\nTarget Exams:\n- Category 1 (Lower Primary)\n- Category 2 (Upper Primary)\n- Category 3 (High School)\n- Category 4 (Specialist)\n\nEligibility:\nTTC / D.El.Ed / B.Ed (Based on Category)\n\nKey Features:\n- Subject-wise live HD video classes\n- Recorded video classes\n- Coverage of Pedagogy\n- Child Development\n- Core subjects\n- Daily workouts\n- Previous Year Question (PYQ) discussions\n- Customized study plans\n- Dedicated mentor support\n\nDelivery Mode:\nCC Learning App (Android & iOS)\n\n--------------------------------------------------------------------------------\n\n================================================================================\nCOURSE NAME: 

In [4]:
# 2. NAIVE CHUNKING
text_splitter = CharacterTextSplitter(chunk_size=700, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
len(chunks)
chunks

[Document(metadata={'source': 'knowledge_base.txt'}, page_content='================================================================================\nCOURSE NAME: KTET (Kerala Teacher Eligibility Test)\n================================================================================\n\nCategory:\nTeaching Eligibility\n\nTarget Exams:\n- Category 1 (Lower Primary)\n- Category 2 (Upper Primary)\n- Category 3 (High School)\n- Category 4 (Specialist)\n\nEligibility:\nTTC / D.El.Ed / B.Ed (Based on Category)\n\nKey Features:\n- Subject-wise live HD video classes\n- Recorded video classes\n- Coverage of Pedagogy\n- Child Development\n- Core subjects\n- Daily workouts\n- Previous Year Question (PYQ) discussions\n- Customized study plans\n- Dedicated mentor support'),
 Document(metadata={'source': 'knowledge_base.txt'}, page_content='Delivery Mode:\nCC Learning App (Android & iOS)\n\n--------------------------------------------------------------------------------\n\n============================

In [5]:
# 3. EMBEDDINGS SETUP
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://127.0.0.1:11434"
)

C:\Users\Nihala\AppData\Local\Temp\ipykernel_14100\1232969550.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [6]:
# 4. VECTOR DATABASE (CHROMADB)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_data"
)

In [7]:
# 5. LLM SETUP
llm = Ollama(
    model="llama3",
    base_url="http://127.0.0.1:11434"
)

C:\Users\Nihala\AppData\Local\Temp\ipykernel_14100\468910716.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [8]:
# 6. CREATE THE RAG CHAIN
qa_chain = RetrievalQA.from_chain_type(
    llm = Ollama(model="llama3"),
    retriever=vector_db.as_retriever(),
    chain_type="stuff"
)

In [ ]:
# 7. QUESTION & ANSWER INTERFACE
print("RAG System Ready! Type 'exit' to quit.")
while True:
    question = input("\nQuestion: ")
    
    if question.lower() == 'exit':
        print('Good Bye')
        break
        
    # Searches ChromaDB for relevant chunks, sends them to Gemma, and generates an answer
    answer = qa_chain.invoke(question)
    
    print(f"Answer: {answer['result']}")

RAG System Ready! Type 'exit' to quit.
